In [42]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (77108, 36)
Test shape: (17014, 35)


In [43]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

In [44]:
print("=== Missing Values in Date Columns ===")
print(train[date_cols].isna().sum())

=== Missing Values in Date Columns ===
order_purchase_timestamp          0
order_approved_at                12
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
dtype: int64


In [45]:
for col in date_cols:
    train[col] = pd.to_datetime(train[col])
    test[col]  = pd.to_datetime(test[col])

print("✅ Date columns converted")
print(train[date_cols].dtypes)

✅ Date columns converted
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [46]:
train['is_late'] = (
    train['order_delivered_customer_date'] > train['order_estimated_delivery_date']
).astype(int)

# Mark missing delivery dates as NaN
missing_delivery = train['order_delivered_customer_date'].isna()
train.loc[missing_delivery, 'is_late'] = np.nan

print("=== is_late created ===")
print(train['is_late'].value_counts(dropna=False))

=== is_late created ===
is_late
0.0    70958
1.0     6150
Name: count, dtype: int64


In [47]:
# See what order_status those missing rows have
print(train[missing_delivery]['order_status'].value_counts())
before = train.shape[0]
train = train.dropna(subset=['order_delivered_customer_date'])
after = train.shape[0]

print(f"\n✅ Undelivered orders dropped")
print(f"Removed: {before - after} rows")
print(f"Remaining rows: {after}")
print(f"Train shape after drop: {train.shape}")

Series([], Name: count, dtype: int64)

✅ Undelivered orders dropped
Removed: 0 rows
Remaining rows: 77108
Train shape after drop: (77108, 36)


In [48]:
counts = train['is_late'].value_counts()
pct = train['is_late'].value_counts(normalize=True) * 100
print(f"Late    (1): {counts[1]} rows  ({pct[1]:.1f}%)")
print(f"On time (0): {counts[0]} rows  ({pct[0]:.1f}%)")

Late    (1): 6150 rows  (8.0%)
On time (0): 70958 rows  (92.0%)


In [49]:
leaky_columns = [
    'order_delivered_customer_date',
    'order_delivered_carrier_date',
    'order_status'
]

print("=== LEAKY COLUMNS — NEVER USE AS FEATURES ===")
for col in leaky_columns:
    print(f"  don't use {col}")

=== LEAKY COLUMNS — NEVER USE AS FEATURES ===
  don't use order_delivered_customer_date
  don't use order_delivered_carrier_date
  don't use order_status


# Basic Feature Engineering

In [50]:
# Create purchase month feature

train['purchase_month'] = train['order_purchase_timestamp'].dt.month
test['purchase_month'] = test['order_purchase_timestamp'].dt.month

In [51]:
train[['order_purchase_timestamp', 'purchase_month']].head()

,order_purchase_timestamp,purchase_month
0,2018-04-30 19:56:03,4
1,2018-03-11 19:30:33,3
2,2018-03-02 15:27:23,3
3,2018-08-23 19:28:09,8
4,2018-04-16 13:14:09,4


In [52]:
#orders placed in each month
train['purchase_month'].value_counts().sort_index()

purchase_month
1     6271
2     6548
3     7619
4     7266
5     8376
6     7366
7     7894
8     8353
9     3370
10    3882
11    5892
12    4271
Name: count, dtype: int64

In [53]:
# Create purchase day feature

train['purchase_day'] = train['order_purchase_timestamp'].dt.day_name()
test['purchase_day'] = test['order_purchase_timestamp'].dt.day_name()

In [54]:
train[['order_purchase_timestamp', 'purchase_day']].head()

,order_purchase_timestamp,purchase_day
0,2018-04-30 19:56:03,Monday
1,2018-03-11 19:30:33,Sunday
2,2018-03-02 15:27:23,Friday
3,2018-08-23 19:28:09,Thursday
4,2018-04-16 13:14:09,Monday


In [55]:
#orders placed in each day
train['purchase_day'].value_counts()

purchase_day
Tuesday      12640
Monday       12638
Wednesday    12090
Thursday     11549
Friday       10860
Sunday        9014
Saturday      8317
Name: count, dtype: int64

In [56]:
# Create weekend feature

train['is_weekend'] = train['purchase_day'].isin(['Saturday', 'Sunday']).astype(int)

test['is_weekend'] = test['purchase_day'].isin(['Saturday', 'Sunday']).astype(int)

In [57]:
train[['purchase_day', 'is_weekend']].head(10)

,purchase_day,is_weekend
0,Monday,0
1,Sunday,1
2,Friday,0
3,Thursday,0
4,Monday,0
5,Sunday,1
6,Monday,0
7,Thursday,0
8,Thursday,0
9,Wednesday,0


In [58]:
#orders placed in weekend and no
train['is_weekend'].value_counts()

is_weekend
0    59777
1    17331
Name: count, dtype: int64

In [59]:
# Create estimated delivery days feature

train['estimated_delivery_days'] = (
    train['order_estimated_delivery_date'] -
    train['order_purchase_timestamp']
).dt.days

test['estimated_delivery_days'] = (
    test['order_estimated_delivery_date'] -
    test['order_purchase_timestamp']
).dt.days

In [60]:
train[
    [
        'order_purchase_timestamp',
        'order_estimated_delivery_date',
        'estimated_delivery_days'
    ]
].head()

,order_purchase_timestamp,order_estimated_delivery_date,estimated_delivery_days
0,2018-04-30 19:56:03,2018-05-29,28
1,2018-03-11 19:30:33,2018-04-03,22
2,2018-03-02 15:27:23,2018-03-28,25
3,2018-08-23 19:28:09,2018-09-18,25
4,2018-04-16 13:14:09,2018-05-02,15


In [61]:
train['estimated_delivery_days'].describe()

count    77108.000000
mean        23.420929
std          8.837262
min          2.000000
25%         18.000000
50%         23.000000
75%         28.000000
max        155.000000
Name: estimated_delivery_days, dtype: float64

In [62]:
train['estimated_delivery_days'].value_counts().sort_index().head(20)

estimated_delivery_days
2      144
3      112
4      255
5      233
6      261
7      689
8      539
9      917
10     995
11    1359
12    2056
13    2012
14    1620
15    2053
16    2246
17    2514
18    2738
19    3736
20    3779
21    4612
Name: count, dtype: int64

In [63]:
import numpy as np

# Create freight-to-price ratio feature

train['freight_to_price_ratio'] = np.where(
    train['price'] != 0,
    train['freight_value'] / train['price'],
    0
)

test['freight_to_price_ratio'] = np.where(
    test['price'] != 0,
    test['freight_value'] / test['price'],
    0
)

In [64]:
train[['price','freight_value','freight_to_price_ratio']].head()

,price,freight_value,freight_to_price_ratio
0,119.00,19.74,0.165882
1,44.90,22.93,0.510690
2,65.90,16.90,0.256449
3,14.49,18.23,1.258109
4,109.99,23.25,0.211383


In [65]:
train['freight_to_price_ratio'].describe()

count    77108.000000
mean         0.320523
std          0.337725
min          0.000000
25%          0.134832
50%          0.231701
75%          0.394874
max         26.235294
Name: freight_to_price_ratio, dtype: float64

In [66]:
# Create same_state feature

train['same_state'] = (
    train['seller_state'] == train['customer_state']
).astype(int)

test['same_state'] = (
    test['seller_state'] == test['customer_state']
).astype(int)

In [67]:
train[['seller_state','customer_state','same_state']].head(10)

,seller_state,customer_state,same_state
0,MG,ES,0
1,SP,MG,0
2,SC,RJ,0
3,SP,RJ,0
4,SP,SP,1
5,SP,ES,0
6,MG,SP,0
7,SP,SC,0
8,SP,SP,1
9,SC,RJ,0


In [68]:
train['same_state'].value_counts()

same_state
0    49176
1    27932
Name: count, dtype: int64

In [69]:
train.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,seller_zip_code_prefix,seller_city,seller_state,is_late,purchase_month,purchase_day,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state
0,249e44947c66e3b0e46b3ba2f38d846d,22920e925fcbdd68ef776651f8c53cc3,delivered,2018-04-30 19:56:03,2018-05-01 02:55:24,2018-05-02 12:20:00,2018-05-09 20:14:48,2018-05-29,1.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,37175.0,ilicinea,MG,0.0,4,Monday,0,28,0.165882,0
1,99e4c7eebc4e218f0ef4729921038c19,7b7d563426a10b8b26bfc679be70fbd8,delivered,2018-03-11 19:30:33,2018-03-11 19:47:54,2018-03-13 21:36:41,2018-04-08 13:37:48,2018-04-03,1.0,d48bacc1dcd9c86bf1ed4ed2a303336c,...,18500.0,tatui,SP,1.0,3,Sunday,1,22,0.510690,0
2,be754126110440ba89a5456475333453,24c2d36dcc7f3ff6e41738c77dcf2def,delivered,2018-03-02 15:27:23,2018-03-06 03:55:55,2018-03-20 15:37:48,2018-03-29 19:21:58,2018-03-28,5.0,d6e74e35591c053e5cbab04d84c223b5,...,88308.0,itajai,SC,1.0,3,Friday,0,25,0.256449,0
3,f6c9fe3ff737f5568e352b5b2afcf112,b354952c4607431ab98d03af79f6d967,delivered,2018-08-23 19:28:09,2018-08-24 19:25:09,2018-08-27 14:46:00,2018-08-30 16:58:52,2018-09-18,1.0,67bd616e1ba0d3d3e8545f3113b0140d,...,11701.0,praia grande,SP,0.0,8,Thursday,0,25,1.258109,0
4,023669233121f0fb7899e5be2b22885f,22c15b46adce8afdd7d58e6582752263,delivered,2018-04-16 13:14:09,2018-04-17 04:54:27,2018-04-18 20:23:49,2018-04-19 16:37:46,2018-05-02,1.0,6b6b162b177d0f36987993aecbe1c65f,...,3916.0,sao paulo,SP,0.0,4,Monday,0,15,0.211383,1


In [70]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77108 entries, 0 to 77107
Data columns (total 36 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       77108 non-null  object        
 1   customer_id                    77108 non-null  object        
 2   order_status                   77108 non-null  object        
 3   order_purchase_timestamp       77108 non-null  datetime64[ns]
 4   order_approved_at              77096 non-null  datetime64[ns]
 5   order_delivered_carrier_date   77107 non-null  datetime64[ns]
 6   order_delivered_customer_date  77108 non-null  datetime64[ns]
 7   order_estimated_delivery_date  77108 non-null  datetime64[ns]
 8   order_item_id                  77108 non-null  float64       
 9   product_id                     77108 non-null  object        
 10  seller_id                      77108 non-null  object        
 11  shipping_limit_

In [71]:
train.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'is_late',
       'purchase_month', 'purchase_day', 'is_weekend',
       'estimated_delivery_days', 'freight_to_price_ratio', 'same_state'],
      dtype='object')

#validation


In [72]:
new_features = [
    'purchase_month',
    'purchase_day',
    'is_weekend',
    'estimated_delivery_days',
    'freight_to_price_ratio',
    'same_state'
]

train[new_features].head()

,purchase_month,purchase_day,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state
0,4,Monday,0,28,0.165882,0
1,3,Sunday,1,22,0.510690,0
2,3,Friday,0,25,0.256449,0
3,8,Thursday,0,25,1.258109,0
4,4,Monday,0,15,0.211383,1


In [73]:
train[new_features].isnull().sum()

purchase_month             0
purchase_day               0
is_weekend                 0
estimated_delivery_days    0
freight_to_price_ratio     0
same_state                 0
dtype: int64

In [74]:
train[new_features].dtypes


purchase_month               int32
purchase_day                object
is_weekend                   int64
estimated_delivery_days      int64
freight_to_price_ratio     float64
same_state                   int64
dtype: object

In [75]:
train[['purchase_month',
       'purchase_day',
       'is_weekend',
       'estimated_delivery_days',
       'freight_to_price_ratio',
       'same_state']].describe(include='all')

,purchase_month,purchase_day,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state
count,77108.000000,77108,77108.000000,77108.000000,77108.000000,77108.000000
unique,NaN,7,NaN,NaN,NaN,NaN
top,NaN,Tuesday,NaN,NaN,NaN,NaN
freq,NaN,12640,NaN,NaN,NaN,NaN
mean,6.026093,NaN,0.224763,23.420929,0.320523,0.362245
std,3.225693,NaN,0.417429,8.837262,0.337725,0.480652
min,1.000000,NaN,0.000000,2.000000,0.000000,0.000000
25%,3.000000,NaN,0.000000,18.000000,0.134832,0.000000
50%,6.000000,NaN,0.000000,23.000000,0.231701,0.000000
75%,8.000000,NaN,0.000000,28.000000,0.394874,1.000000


In [76]:
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)